In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import optuna

# Team name mapping dictionary
team_mapping = {
    'Man Utd': 'Manchester Utd',
    'Man United': 'Manchester Utd',
    'Man City': 'Manchester City',
    'Newcastle Utd': 'Newcastle United',
    'Newcastle Ut': 'Newcastle United',
    "Nott'ham Forest": 'Nottingham Forest',
    'Paris S-G': 'Paris Saint-Germain',
    'Inter Milan': 'Inter',
    'Spurs': 'Tottenham',
    'Man Utd': 'Manchester United',
    'West Ham Utd': 'West Ham United'
}

def standardize_team_names(df, column_name):
    df[column_name] = df[column_name].replace(team_mapping).str.strip()
    return df

# Load and preprocess data
stats_df = pd.read_csv('Combined_Leagues_Stats.csv')
fixtures_df = pd.read_csv('Fixture Results.csv')

# Apply team name standardization
stats_df = standardize_team_names(stats_df, 'team')
fixtures_df = standardize_team_names(fixtures_df, 'Home_Team')
fixtures_df = standardize_team_names(fixtures_df, 'Away_Team')

# Feature engineering
stats_df['total_progression'] = stats_df['progressive_carries'] + stats_df['progressive_passes']

feature_columns = [
    'team', 'cards_yellow', 'cards_red', 'xg', 'npxg', 'xg_assist', 'npxg_xg_assist',
    'progressive_carries', 'progressive_passes', 'total_progression', 'goals_per90',
    'assists_per90', 'goals_assists_per90', 'goals_pens_per90', 'goals_assists_pens_per90',
    'xg_per90', 'xg_assist_per90', 'xg_xg_assist_per90', 'npxg_per90', 'npxg_xg_assist_per90'
]

stats_df = stats_df[feature_columns].fillna(0)
team_stats = stats_df.set_index('team').to_dict('index')

# Feature extraction function
def get_features(row):
    home_team = row['Home_Team'].strip()
    away_team = row['Away_Team'].strip()
    
    # Get features with fallback to empty features
    home_features = list(team_stats.get(home_team, {}).values())[1:]  # Exclude team name
    away_features = list(team_stats.get(away_team, {}).values())[1:]  # Exclude team name
    
    # Handle missing teams
    if not home_features:
        print(f"Warning: No data found for home team {home_team}")
        home_features = [0] * (len(feature_columns)-1)
    if not away_features:
        print(f"Warning: No data found for away team {away_team}")
        away_features = [0] * (len(feature_columns)-1)
    
    return home_features + away_features

# Prepare targets
fixtures_df['features'] = fixtures_df.apply(get_features, axis=1)
fixtures_df['result'] = fixtures_df.apply(
    lambda row: 2 if row['Home_Score'] > row['Away_Score'] else 1 if row['Home_Score'] == row['Away_Score'] else 0, axis=1
)

# Train-test split
X = np.array(fixtures_df['features'].tolist())
y_class = fixtures_df['result'].values
y_reg = fixtures_df[['Home_xG', 'Away_xG']].values
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42
)

# Normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Dataset class
class FootballDataset(Dataset):
    def __init__(self, X, y_class, y_reg):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_class = torch.tensor(y_class, dtype=torch.long)
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y_class[idx], self.y_reg[idx]

train_dataset = FootballDataset(X_train, y_class_train, y_reg_train)
test_dataset = FootballDataset(X_test, y_class_test, y_reg_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Model architecture
class FootballPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.base = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.class_head = nn.Linear(hidden_dim, 3)  # Home Win, Draw, Away Win
        self.reg_head = nn.Linear(hidden_dim, 2)    # Home xG, Away xG
    
    def forward(self, x):
        x = self.base(x)
        return self.class_head(x), self.reg_head(x)

# Hyperparameter tuning with Optuna
def objective(trial):
    hidden_dim = trial.suggest_int('hidden_dim', 32, 128)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    model = FootballPredictor(X_train.shape[1], hidden_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion_class = nn.CrossEntropyLoss()
    criterion_reg = nn.MSELoss()
    
    for epoch in range(20):
        model.train()
        for X_batch, y_class_batch, y_reg_batch in train_loader:
            optimizer.zero_grad()
            class_out, reg_out = model(X_batch)
            loss = criterion_class(class_out, y_class_batch) + criterion_reg(reg_out, y_reg_batch)
            loss.backward()
            optimizer.step()
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_class_batch, y_reg_batch in test_loader:
            class_out, reg_out = model(X_batch)
            val_loss += (criterion_class(class_out, y_class_batch) + criterion_reg(reg_out, y_reg_batch)).item()
    return val_loss / len(test_loader)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

# Train final model
best_params = study.best_params
model = FootballPredictor(X_train.shape[1], best_params['hidden_dim'])
optimizer = optim.Adam(model.parameters(), lr=best_params['lr'])
criterion_class = nn.CrossEntropyLoss()
criterion_reg = nn.MSELoss()

for epoch in range(100):
    model.train()
    total_loss = 0
    for X_batch, y_class_batch, y_reg_batch in train_loader:
        optimizer.zero_grad()
        class_out, reg_out = model(X_batch)
        loss = criterion_class(class_out, y_class_batch) + criterion_reg(reg_out, y_reg_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_class_batch, y_reg_batch in test_loader:
            class_out, reg_out = model(X_batch)
            val_loss += (criterion_class(class_out, y_class_batch) + criterion_reg(reg_out, y_reg_batch)).item() * len(X_batch)
    
    print(f'Epoch {epoch+1}, Loss: {total_loss/len(train_dataset):.4f}, Val Loss: {val_loss/len(test_dataset):.4f}')

# Prediction function with xG output
def predict_fixtures(model, scaler, fixtures_csv_path):
    new_fixtures = pd.read_csv(fixtures_csv_path)
    new_fixtures = standardize_team_names(new_fixtures, 'Home_Team')
    new_fixtures = standardize_team_names(new_fixtures, 'Away_Team')
    
    new_fixtures['features'] = new_fixtures.apply(get_features, axis=1)
    X_new = scaler.transform(np.array(new_fixtures['features'].tolist()))
    X_new = torch.tensor(X_new, dtype=torch.float32)
    
    model.eval()
    with torch.no_grad():
        class_logits, reg_preds = model(X_new)
        class_probs = torch.softmax(class_logits, dim=1).numpy()
    
    return pd.DataFrame({
        'Home_Team': new_fixtures['Home_Team'],
        'Away_Team': new_fixtures['Away_Team'],
        'Home_Win_Prob': class_probs[:, 2],
        'Draw_Prob': class_probs[:, 1],
        'Away_Win_Prob': class_probs[:, 0],
        'Predicted_Home_xG': reg_preds[:, 0].numpy(),
        'Predicted_Away_xG': reg_preds[:, 1].numpy()
    })

# Example usage
# predictions = predict_fixtures(model, scaler, 'fixtures.csv')
# print(predictions[['Home_Team', 'Away_Team', 'Home_Win_Prob', 'Draw_Prob', 'Away_Win_Prob', 
#                   'Predicted_Home_xG', 'Predicted_Away_xG']])

[I 2025-02-09 00:37:15,518] A new study created in memory with name: no-name-fb1044c6-686c-4639-a13b-64b9119cbe69
[I 2025-02-09 00:37:18,809] Trial 0 finished with value: 1.5372113314541904 and parameters: {'hidden_dim': 66, 'lr': 0.00019872175472275333}. Best is trial 0 with value: 1.5372113314541904.
[I 2025-02-09 00:37:20,783] Trial 1 finished with value: 1.6080996665087612 and parameters: {'hidden_dim': 57, 'lr': 0.006333565128074568}. Best is trial 0 with value: 1.5372113314541904.
[I 2025-02-09 00:37:22,651] Trial 2 finished with value: 1.5379764600233599 and parameters: {'hidden_dim': 35, 'lr': 0.00047756790304495254}. Best is trial 0 with value: 1.5372113314541904.
[I 2025-02-09 00:37:24,469] Trial 3 finished with value: 1.5379221547733655 and parameters: {'hidden_dim': 60, 'lr': 0.0002174717560450479}. Best is trial 0 with value: 1.5372113314541904.
[I 2025-02-09 00:37:26,319] Trial 4 finished with value: 1.5558710206638684 and parameters: {'hidden_dim': 100, 'lr': 0.000643033

Epoch 1, Loss: 2.5525, Val Loss: 1.8645
Epoch 2, Loss: 1.7345, Val Loss: 1.7003
Epoch 3, Loss: 1.6358, Val Loss: 1.6271
Epoch 4, Loss: 1.5886, Val Loss: 1.5959
Epoch 5, Loss: 1.5499, Val Loss: 1.5821
Epoch 6, Loss: 1.5281, Val Loss: 1.5610
Epoch 7, Loss: 1.5031, Val Loss: 1.5442
Epoch 8, Loss: 1.4878, Val Loss: 1.5475
Epoch 9, Loss: 1.4760, Val Loss: 1.5360
Epoch 10, Loss: 1.4623, Val Loss: 1.5395
Epoch 11, Loss: 1.4552, Val Loss: 1.5310
Epoch 12, Loss: 1.4434, Val Loss: 1.5316
Epoch 13, Loss: 1.4399, Val Loss: 1.5309
Epoch 14, Loss: 1.4326, Val Loss: 1.5406
Epoch 15, Loss: 1.4303, Val Loss: 1.5315
Epoch 16, Loss: 1.4260, Val Loss: 1.5295
Epoch 17, Loss: 1.4216, Val Loss: 1.5450
Epoch 18, Loss: 1.4216, Val Loss: 1.5590
Epoch 19, Loss: 1.4167, Val Loss: 1.5373
Epoch 20, Loss: 1.4144, Val Loss: 1.5305
Epoch 21, Loss: 1.4138, Val Loss: 1.5376
Epoch 22, Loss: 1.4053, Val Loss: 1.5346
Epoch 23, Loss: 1.4097, Val Loss: 1.5478
Epoch 24, Loss: 1.4007, Val Loss: 1.5459
Epoch 25, Loss: 1.4054, V

In [3]:
predictions = predict_fixtures(model, scaler, 'fixtures.csv')
print(predictions[['Home_Team', 'Away_Team', 'Home_Win_Prob', 'Draw_Prob', 'Away_Win_Prob', 'Predicted_Home_xG', 'Predicted_Away_xG']])

       Home_Team        Away_Team  Home_Win_Prob  Draw_Prob  Away_Win_Prob  \
0      West Brom   Sheffield Weds       0.452369   0.213794       0.333838   
1     Sunderland          Watford       0.607547   0.304962       0.087490   
2   Norwich City     Derby County       0.769298   0.208513       0.022189   
3  Sheffield Utd       Portsmouth       0.825563   0.136972       0.037465   
4     Celta Vigo            Betis       0.400037   0.282852       0.317110   
5  Athletic Club           Girona       0.398941   0.497283       0.103776   
6     Las Palmas       Villarreal       0.162697   0.243249       0.594054   
7    Real Madrid  Atlético Madrid       0.748880   0.143490       0.107629   

   Predicted_Home_xG  Predicted_Away_xG  
0           1.300056           1.343196  
1           1.226742           0.820950  
2           2.008776           0.778204  
3           2.068756           0.864783  
4           1.148129           1.184308  
5           1.629257           0.766068  
6  